# Training commands

Select `phase` and `model = list_model[n]`, then run the corresponding cell to print **one shell command**

Copy the output into a **bash** terminal from any directory

※ This notebook does not execute training

- **T1 phases:** `train`, `inference`, `evaluation`
- **T2 phases:** `cache`, `train`, `inference`, `evaluation`
- `epoch_load`: T1 training uses `0` (new run) or `-1` (resume from `last_point.pt`)
- `epoch_eval`: checkpoint used for inference or evaluation

**Execution order:** 

T1 ASTRA Phase 1 → T1 ASTA Phase 2 (requires the Phase 1 teacher) 

## T1 — ASTRA-Net and comparison models

In [2]:
import shlex
from pathlib import Path

In [4]:
GPU_NUM = 0
phase = 'train'
# phase = 'inference'
# phase = 'evaluation'
name_dset = 'Preset1'
list_model = [
    'UNet',
    'ResUNet',
    'FANN',
    'TSE_Net',
    'UDADS_Net',
    'WingsNet',
    'ASTRA_Net_phase_1',
    'ASTRA_Net_phase_2']
model = list_model[-1]
epoch_load = 0
epoch_eval = -1
full_train = False
batch_size_inference = 0
n_case = 0
dset_test = []
end = '\\\n'

path_repo = Path.cwd().resolve()
if not (path_repo/'src_T1').is_dir(): path_repo = path_repo.parent

list_arg = [f'python -u src_T1/{phase}.py']
if phase == 'train':
    list_arg += [f'--GPU_NUM {GPU_NUM}', f'--name_dset {shlex.quote(name_dset)}', f'--name_model {shlex.quote(model)}', f'--epoch_load {epoch_load}']
    if full_train: list_arg.append('--full_train')
elif phase in ['inference', 'evaluation']:
    if phase == 'inference': list_arg.append(f'--GPU_NUM {GPU_NUM}')
    if phase == 'inference' and batch_size_inference>0: list_arg.append(f'--batch_size_inference {batch_size_inference}')
    list_arg += [f'--name_dset {shlex.quote(name_dset)}', f'--name_model {shlex.quote(model)}', f'--epoch_load {epoch_eval}']
    if n_case > 0: list_arg.append(f'--n {n_case}')
    if dset_test: list_arg.append('--dset_test ' + ' '.join(shlex.quote(dset) for dset in dset_test))
else:
    raise ValueError(f'Unknown phase: {phase}')

cmd = f'tput rmam && cd {shlex.quote(str(path_repo))} && {end}'
cmd += f' {end}    '.join(list_arg)
cmd += ' ; tput smam'
print(cmd)

tput rmam && cd /nas/Guhong/Airway/src && \
python -u src_T1/train.py \
    --GPU_NUM 0 \
    --name_dset Preset1 \
    --name_model ASTRA_Net_phase_2 \
    --epoch_load 0 ; tput smam


## T2 — ATM26 Reference Baseline 

**outside the scope of our paper**

※ Requires the T1 ASTRA-Net Phase 2 checkpoint

In [3]:
import shlex
from pathlib import Path

GPU_NUM = 0
phase = 'cache'
# phase = 'train'
# phase = 'inference'
# phase = 'evaluation'
name_dset = 'Preset1'
list_model = ['ASTRA_Net']
n = 0
model = list_model[n]
epoch_load = 0
epoch_eval = -1
full_train = False
use_extra_train = False
n_case = 0
end = '\\\n'

path_repo = Path.cwd().resolve()
if not (path_repo/'src_T2').is_dir(): path_repo = path_repo.parent

if phase == 'cache':
    list_arg = ['python -u src_T2/cache_track1.py', f'--GPU_NUM {GPU_NUM}']
elif phase == 'train':
    list_arg = ['python -u src_T2/train.py', f'--GPU_NUM {GPU_NUM}', f'--name_dset {shlex.quote(name_dset)}', f'--name_model {shlex.quote(model)}', f'--epoch_load {epoch_load}']
    if full_train: list_arg.append('--full_train')
    if use_extra_train: list_arg.append('--use_extra_train')
elif phase in ['inference', 'evaluation']:
    list_arg = [f'python -u src_T2/{phase}.py']
    if phase == 'inference': list_arg.append(f'--GPU_NUM {GPU_NUM}')
    list_arg += [f'--name_dset {shlex.quote(name_dset)}', f'--name_model {shlex.quote(model)}', f'--epoch_load {epoch_eval}']
    if n_case > 0: list_arg.append(f'--n {n_case}')
else:
    raise ValueError(f'Unknown phase: {phase}')

cmd = f'tput rmam && cd {shlex.quote(str(path_repo))} && {end}'
cmd += f' {end}    '.join(list_arg)
cmd += ' ; tput smam'
print(cmd)

tput rmam && cd /nas/Guhong/Airway/src && \
python -u src_T2/cache_track1.py \
    --GPU_NUM 0 ; tput smam
